In [1]:
"""
=============================================================================
PREDIÇÕES COPA DO MUNDO 2026
=============================================================================
Autor: Modelo preditivo baseado em dados reais de 48 seleções
Dados:
  - media_fifa.xlsx        → médias estatísticas recentes de cada seleção
  - fifa_48_selecoes.xlsx  → dados brutos por jogo de cada seleção
  - Estadios_altitudes.xlsx → altitude, temperatura e dados climáticos
  - dados_brutos_temperaturas.xlsx → séries históricas de temperatura
 
Metodologia:
  1. Estatística Descritiva por seleção
  2. Regressão Linear Múltipla para estimar xG esperado por partida
  3. Modelo Poisson para simular placar e probabilidades
  4. Ajuste por altitude e temperatura do estádio
  5. Score composto para ranking e previsão de vencedor
 
Saída: predicoes_copa2026_resultados.xlsx
=============================================================================
"""
 
# =============================================================================
# CONFIGURAÇÕES (ajuste aqui se necessário)
# =============================================================================
ARQUIVO_MEDIA      = 'C:/Users/raphael.eugenio/Desktop/Raphael/WC26/Scripts//media_fifa.xlsx'
ARQUIVO_SELECOES   = 'C:/Users/raphael.eugenio/Desktop/Raphael/WC26/Scripts//fifa_48_selecoes.xlsx'
ARQUIVO_ESTADIOS   = 'C:/Users/raphael.eugenio/Desktop/Raphael/WC26/Scripts//Estadios_altitudes.xlsx'
ARQUIVO_TEMPS      = 'C:/Users/raphael.eugenio/Desktop/Raphael/WC26/Scripts//dados_brutos_temperaturas.xlsx'
ARQUIVO_SAIDA      = 'C:/Users/raphael.eugenio/Desktop/Raphael/WC26/Scripts/predicoes_copa2026_resultados.xlsx'
N_SIMULACOES       = 50000  # Monte Carlo para probabilidades

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import poisson, pearsonr, linregress
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from openpyxl import Workbook
from openpyxl.styles import (Font, PatternFill, Alignment, Border, Side,
                              numbers as xl_numbers)
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows
import warnings
warnings.filterwarnings('ignore')
 
np.random.seed(42)
 
print("=" * 70)
print("PREDIÇÕES COPA DO MUNDO 2026 — Carregando dados...")
print("=" * 70)

PREDIÇÕES COPA DO MUNDO 2026 — Carregando dados...


In [4]:
# =============================================================================
# 1. CARGA E LIMPEZA DOS DADOS
# =============================================================================
 
# --- 1a. Médias por seleção (Planilha1 de media_fifa) ---
df_raw = pd.read_excel(ARQUIVO_MEDIA, sheet_name='Planilha1', index_col=0)
df_teams = df_raw.T.copy()
df_teams.index.name = 'Selecao'
df_teams = df_teams.loc[df_teams.index != 'Continente']
for col in df_teams.columns:
    df_teams[col] = pd.to_numeric(df_teams[col], errors='coerce')
 
# Continente para cada seleção
continentes = df_raw.loc['Continente'].to_dict()
 
# --- 1b. Matriz de correlação entre métricas ---
df_corr_raw = pd.read_excel(ARQUIVO_MEDIA, sheet_name='Planilha2', index_col=0)
 
# --- 1c. Estádios: altitude e temperatura ---
df_est = pd.read_excel(ARQUIVO_ESTADIOS, sheet_name='ALL')
stadiums_sheets = [
    'Estádio Azteca','Estádio Akron','Estádio BBVA','BMO Field','BC Place',
    'Estádio Mercedes-Benz','Estádio Gillette','Estádio AT&T',
    'Lincoln Financial Field','Estádio NGR','GEHA Field at Arrowhead',
    'Estádio SoFi','Estádio Hard Rock','Estádio Met Life',
    "Estádio Levi's",'Lumen Field'
]
avg_temps = {}
for s in stadiums_sheets:
    df_s = pd.read_excel(ARQUIVO_ESTADIOS, sheet_name=s)
    t_col = [c for c in df_s.columns if 'temperature_2m' in str(c).lower()
             and 'apparent' not in str(c).lower()]
    if t_col:
        avg_temps[s] = df_s[t_col[0]].mean()
df_est['temp_media_calc'] = df_est['ESTADIO'].map(avg_temps)
 
# Mapeamento cidade → estádio
cidade_estadio = dict(zip(df_est['CIDADE'], df_est['ESTADIO']))
estadio_altitude = dict(zip(df_est['ESTADIO'], df_est['ALTITUDE (m)']))
estadio_temp = dict(zip(df_est['ESTADIO'], df_est['temp_media_calc']))
estadio_vento = dict(zip(df_est['ESTADIO'], df_est['Velocidade do vento a 10 metros']))
 
print(f"✓ {len(df_teams)} seleções carregadas | {df_teams.shape[1]} métricas")
print(f"✓ {len(df_est)} estádios | altitude máx: {df_est['ALTITUDE (m)'].max():.0f}m (Azteca)")

✓ 48 seleções carregadas | 40 métricas
✓ 16 estádios | altitude máx: 2237m (Azteca)


In [ ]:
# 12 grupos (A-L) x 4 seleções x 6 jogos = 72 jogos da fase de grupos
# Nomes das seleções compatíveis com a base media_fifa.xlsx / fifa_48_selecoes.xlsx
 
grupos_oficiais = {
    'A': ['México', 'África do Sul', 'Córeia do Sul', 'Republica Tcheca'],
    'B': ['Canadá', 'Bósnia e Herzegovina', 'Catar', 'Suíça'],
    'C': ['Brasil', 'Marrocos', 'Haiti', 'Escócia'],
    'D': ['Eua', 'Paraguai', 'Austrália', 'Turquia'],
    'E': ['Alemanha', 'Curaçao', 'Costa do Marfim', 'Equador'],
    'F': ['Holanda', 'Japão', 'Suécia', 'Tunísia'],
    'G': ['Bélgica', 'Egito', 'Irã', 'Nova Zelândia'],
    'H': ['Espanha', 'Cabo Verde', 'Arábia Saudita', 'Uruguai'],
    'I': ['França', 'Senegal', 'Iraque', 'Noruega'],
    'J': ['Argentina', 'Argélia', 'Áustria', 'Jordânia'],
    'K': ['Portugal', 'RD Congo', 'Uzbequistão', 'Colômbia'],
    'L': ['Inglaterra', 'Croácia', 'Gana', 'Panamá'],
}
 
# Jogos oficiais da fase de grupos: (Grupo, Time1, Time2, Data, Estádio)
# Datas no formato DD/06. Estádios já com os nomes usados em Estadios_altitudes.xlsx
jogos_oficiais = [
    # GRUPO A — Cidade do México / Guadalajara / Atlanta / Monterrey
    ('A','México','África do Sul','11/06','Estádio Azteca'),
    ('A','Córeia do Sul','Republica Tcheca','11/06','Estádio Akron'),
    ('A','Republica Tcheca','África do Sul','18/06','Estádio Mercedes-Benz'),
    ('A','México','Córeia do Sul','18/06','Estádio Akron'),
    ('A','Republica Tcheca','México','24/06','Estádio Azteca'),
    ('A','África do Sul','Córeia do Sul','24/06','Estádio BBVA'),
 
    # GRUPO B — Toronto / São Francisco / Los Angeles / Vancouver / Seattle
    ('B','Canadá','Bósnia e Herzegovina','12/06','BMO Field'),
    ('B','Catar','Suíça','13/06',"Estádio Levi's"),
    ('B','Suíça','Bósnia e Herzegovina','18/06','Estádio SoFi'),
    ('B','Canadá','Catar','18/06','BC Place'),
    ('B','Suíça','Canadá','24/06','BC Place'),
    ('B','Bósnia e Herzegovina','Catar','24/06','Lumen Field'),
 
    # GRUPO C — Nova York / Boston / Filadélfia / Miami / Atlanta
    ('C','Brasil','Marrocos','13/06','Estádio Met Life'),
    ('C','Haiti','Escócia','13/06','Estádio Gillette'),
    ('C','Escócia','Marrocos','19/06','Estádio Gillette'),
    ('C','Brasil','Haiti','19/06','Lincoln Financial Field'),
    ('C','Escócia','Brasil','24/06','Estádio Hard Rock'),
    ('C','Marrocos','Haiti','24/06','Estádio Mercedes-Benz'),
 
    # GRUPO D — Los Angeles / Vancouver / Seattle / São Francisco
    ('D','Eua','Paraguai','12/06','Estádio SoFi'),
    ('D','Austrália','Turquia','13/06','BC Place'),
    ('D','Eua','Austrália','19/06','Lumen Field'),
    ('D','Turquia','Paraguai','19/06',"Estádio Levi's"),
    ('D','Turquia','Eua','25/06','Estádio SoFi'),
    ('D','Paraguai','Austrália','25/06',"Estádio Levi's"),
 
    # GRUPO E — Houston / Filadélfia / Toronto / Kansas City / Nova York
    ('E','Alemanha','Curaçao','14/06','Estádio NGR'),
    ('E','Costa do Marfim','Equador','14/06','Lincoln Financial Field'),
    ('E','Alemanha','Costa do Marfim','20/06','BMO Field'),
    ('E','Equador','Curaçao','20/06','GEHA Field at Arrowhead'),
    ('E','Equador','Alemanha','25/06','Estádio Met Life'),
    ('E','Curaçao','Costa do Marfim','25/06','Lincoln Financial Field'),
 
    # GRUPO F — Dallas / Monterrey / Houston / Kansas City
    ('F','Holanda','Japão','14/06','Estádio AT&T'),
    ('F','Suécia','Tunísia','14/06','Estádio BBVA'),
    ('F','Holanda','Suécia','20/06','Estádio NGR'),
    ('F','Tunísia','Japão','20/06','Estádio BBVA'),
    ('F','Japão','Suécia','25/06','Estádio AT&T'),
    ('F','Tunísia','Holanda','25/06','GEHA Field at Arrowhead'),
 
    # GRUPO G — Seattle / Los Angeles / Vancouver
    ('G','Bélgica','Egito','15/06','Lumen Field'),
    ('G','Irã','Nova Zelândia','15/06','Estádio SoFi'),
    ('G','Bélgica','Irã','21/06','Estádio SoFi'),
    ('G','Nova Zelândia','Egito','21/06','BC Place'),
    ('G','Egito','Irã','26/06','Lumen Field'),
    ('G','Nova Zelândia','Bélgica','26/06','BC Place'),
 
    # GRUPO H — Atlanta / Miami / Houston / Guadalajara
    ('H','Espanha','Cabo Verde','15/06','Estádio Mercedes-Benz'),
    ('H','Arábia Saudita','Uruguai','15/06','Estádio Hard Rock'),
    ('H','Espanha','Arábia Saudita','21/06','Estádio Mercedes-Benz'),
    ('H','Uruguai','Cabo Verde','21/06','Estádio Hard Rock'),
    ('H','Cabo Verde','Arábia Saudita','26/06','Estádio NGR'),
    ('H','Uruguai','Espanha','26/06','Estádio Akron'),
 
    # GRUPO I — Nova York / Boston / Filadélfia / Toronto
    ('I','França','Senegal','16/06','Estádio Met Life'),
    ('I','Iraque','Noruega','16/06','Estádio Gillette'),
    ('I','França','Iraque','22/06','Lincoln Financial Field'),
    ('I','Noruega','Senegal','22/06','Estádio Met Life'),
    ('I','Noruega','França','26/06','Estádio Gillette'),
    ('I','Senegal','Iraque','26/06','BMO Field'),
 
    # GRUPO J — Kansas City / São Francisco / Dallas
    ('J','Argentina','Argélia','16/06','GEHA Field at Arrowhead'),
    ('J','Áustria','Jordânia','16/06',"Estádio Levi's"),
    ('J','Argentina','Áustria','22/06','Estádio AT&T'),
    ('J','Jordânia','Argélia','22/06',"Estádio Levi's"),
    ('J','Argélia','Áustria','27/06','GEHA Field at Arrowhead'),
    ('J','Jordânia','Argentina','27/06','Estádio AT&T'),
 
    # GRUPO K — Houston / Cidade do México / Guadalajara / Miami / Atlanta
    ('K','Portugal','RD Congo','17/06','Estádio NGR'),
    ('K','Uzbequistão','Colômbia','17/06','Estádio Azteca'),
    ('K','Portugal','Uzbequistão','23/06','Estádio NGR'),
    ('K','Colômbia','RD Congo','23/06','Estádio Akron'),
    ('K','Colômbia','Portugal','27/06','Estádio Hard Rock'),
    ('K','RD Congo','Uzbequistão','27/06','Estádio Mercedes-Benz'),
 
    # GRUPO L — Dallas / Toronto / Boston / Nova York / Filadélfia
    ('L','Inglaterra','Croácia','17/06','Estádio AT&T'),
    ('L','Gana','Panamá','17/06','BMO Field'),
    ('L','Inglaterra','Gana','23/06','Estádio Gillette'),
    ('L','Panamá','Croácia','23/06','BMO Field'),
    ('L','Panamá','Inglaterra','27/06','Estádio Met Life'),
    ('L','Croácia','Gana','27/06','Lincoln Financial Field'),
]
 
df_jogos = pd.DataFrame(jogos_oficiais,
    columns=['Grupo','Time1','Time2','Data','Estadio'])
 
# Mapear estádio -> cidade (para exibição)
estadio_cidade = dict(zip(df_est['ESTADIO'], df_est['CIDADE']))
df_jogos['Cidade'] = df_jogos['Estadio'].map(estadio_cidade)
 
df_jogos['Altitude'] = df_jogos['Estadio'].map(estadio_altitude)
df_jogos['Temp_Media'] = df_jogos['Estadio'].map(estadio_temp)
df_jogos['Vento'] = df_jogos['Estadio'].map(estadio_vento)
 
# Validação: checar se todos os times existem na base e se há valores nulos
times_jogos = set(df_jogos['Time1']) | set(df_jogos['Time2'])
times_faltando = times_jogos - set(df_teams.index)
if times_faltando:
    print(f"⚠ Times não encontrados na base de médias: {times_faltando}")
 
estadios_faltando = df_jogos[df_jogos['Estadio'].isna() | df_jogos['Altitude'].isna()]
if not estadios_faltando.empty:
    print(f"⚠ Estádios sem dados de altitude/temperatura: {estadios_faltando['Estadio'].unique()}")
 
print(f"✓ {len(df_jogos)} jogos da fase de grupos carregados (12 grupos x 6 jogos)")
print(f"✓ {len(times_jogos)} seleções únicas presentes na fase de grupos")

✓ 72 jogos na fase de grupos carregados


In [6]:
# =============================================================================
# 3. ESTATÍSTICA DESCRITIVA
# =============================================================================
 
metricas_desc = ['Gols esperados (xG)', 'xG sofridos (xGC)', 'Chances perigosas criadas',
                 'Posse de Bola', 'Passes completos', 'Ataque', 'Total de chutes',
                 'Chutes no gol', 'Interceptações', 'Duelos ganhos']
 
df_desc = df_teams[metricas_desc].describe().T
df_desc.columns = ['N','Média','Std','Min','Q25','Mediana','Q75','Max']
df_desc = df_desc.round(3)
 
print("\n--- ESTATÍSTICA DESCRITIVA (amostra) ---")
print(df_desc[['Média','Std','Min','Max']].to_string())
 


--- ESTATÍSTICA DESCRITIVA (amostra) ---
                             Média      Std     Min      Max
Paises                                                      
Gols esperados (xG)          1.342    0.794   0.000    2.855
xG sofridos (xGC)            0.868    0.448   0.000    1.950
Chances perigosas criadas    2.468    1.157   0.000    5.900
Posse de Bola                0.551    0.076   0.387    0.697
Passes completos           398.774  117.721   0.000  644.600
Ataque                      81.301   21.344  12.000  128.800
Total de chutes             12.742    3.720   6.500   21.000
Chutes no gol                4.770    1.414   2.500    8.600
Interceptações               7.714    2.022   0.000   13.000
Duelos ganhos               44.197    9.330   0.000   74.000


In [7]:
# =============================================================================
# 4. REGRESSÃO LINEAR MÚLTIPLA (xG ~ features de ataque)
# =============================================================================
 
features_ataque = [
    'Chances perigosas criadas', 'Chutes no gol', 'Passes decisivos',
    'Assistência esperada (xA)', 'Passes no último terço', 'Posse de Bola',
    'Total de chutes', 'Chutes dentro da área'
]
target_ataque = 'Gols esperados (xG)'
target_defesa = 'xG sofridos (xGC)'
 
df_model = df_teams[features_ataque + [target_ataque, target_defesa]].dropna()
 
X_atq = df_model[features_ataque].values
y_atq = df_model[target_ataque].values
y_def = df_model[target_defesa].values
 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_atq)
 
reg_atq = Ridge(alpha=1.0)
reg_atq.fit(X_scaled, y_atq)
 
reg_def = Ridge(alpha=1.0)
reg_def.fit(X_scaled, y_def)
 
y_pred_atq = reg_atq.predict(X_scaled)
y_pred_def = reg_def.predict(X_scaled)
 
r2_atq = r2_score(y_atq, y_pred_atq)
r2_def = r2_score(y_def, y_pred_def)
 
print(f"\n--- REGRESSÃO LINEAR (Ridge) ---")
print(f"  R² xG Ataque: {r2_atq:.4f}")
print(f"  R² xGC Defesa: {r2_def:.4f}")
 
# Coeficientes
df_coef = pd.DataFrame({
    'Feature': features_ataque,
    'Coef_xG':  reg_atq.coef_.round(4),
    'Coef_xGC': reg_def.coef_.round(4),
})
 


--- REGRESSÃO LINEAR (Ridge) ---
  R² xG Ataque: 0.9486
  R² xGC Defesa: 0.3728


In [8]:
# =============================================================================
# 5. SCORE OFENSIVO E DEFENSIVO POR SELEÇÃO
# =============================================================================
 
def compute_team_score(selecao):
    if selecao not in df_teams.index:
        # fallback: média global
        return df_teams[target_ataque].mean(), df_teams[target_defesa].mean()
    row = df_teams.loc[selecao]
    feats = []
    for f in features_ataque:
        v = row.get(f, np.nan)
        if pd.isna(v):
            v = df_teams[f].median()
        feats.append(v)
    x = scaler.transform([feats])
    xg_pred = max(reg_atq.predict(x)[0], 0.2)
    xgc_pred = max(reg_def.predict(x)[0], 0.2)
    return xg_pred, xgc_pred
 
# Pre-calcular para todas
team_xg  = {}
team_xgc = {}
for t in df_teams.index:
    xg, xgc = compute_team_score(t)
    team_xg[t]  = xg
    team_xgc[t] = xgc

In [9]:
# =============================================================================
# 6. AJUSTE AMBIENTAL (altitude + temperatura)
# =============================================================================
# Altitude >1500m penaliza times de nível do mar em ~5%
# Temperatura >28°C ou <10°C penaliza ambos levemente
 
def ajuste_ambiental(altitude, temperatura):
    """Retorna fator multiplicador para xG (leve redução em condições extremas)"""
    fator = 1.0
    if altitude and altitude > 1500:
        fator *= 0.97   # alta altitude: jogo mais lento, menos gols
    if temperatura:
        if temperatura > 28:
            fator *= 0.97  # calor extremo reduz intensidade
        elif temperatura < 10:
            fator *= 0.98
    return fator

In [14]:
# =============================================================================
# 7. MODELO POISSON + MONTE CARLO
# =============================================================================
 
def prever_jogo(time1, time2, altitude=5, temperatura=20, n_sim=N_SIMULACOES):
    """
    Retorna dicionário com:
      - lambda1, lambda2: gols esperados
      - prob_v1, prob_e, prob_v2: probabilidades
      - placar_mais_provavel: (g1, g2)
      - vencedor: nome ou 'Empate'
    """
    xg1, xgc1 = compute_team_score(time1)
    xg2, xgc2 = compute_team_score(time2)
 
    # Média harmônica: ataque de T1 vs defesa de T2
    lam1 = (xg1 + (1 - xgc2 / df_teams[target_defesa].mean())) / 2
    lam2 = (xg2 + (1 - xgc1 / df_teams[target_defesa].mean())) / 2
    lam1 = max(lam1, 0.15)
    lam2 = max(lam2, 0.15)
 
    # Ajuste ambiental
    fa = ajuste_ambiental(altitude, temperatura)
    lam1 *= fa
    lam2 *= fa
 
    # Simulação Monte Carlo
    g1 = np.random.poisson(lam1, n_sim)
    g2 = np.random.poisson(lam2, n_sim)
 
    prob_v1 = (g1 > g2).mean()
    prob_e  = (g1 == g2).mean()
    prob_v2 = (g1 < g2).mean()
 
    # Placar mais provável (modo)
    from collections import Counter
    placares = Counter(zip(g1.tolist(), g2.tolist()))
    placar_top = placares.most_common(3)
 
    if prob_v1 >= prob_v2 and prob_v1 >= prob_e:
        vencedor = time1
    elif prob_v2 >= prob_v1 and prob_v2 >= prob_e:
        vencedor = time2
    else:
        vencedor = 'Empate'
 
    return {
        'lambda1': round(lam1, 3),
        'lambda2': round(lam2, 3),
        'prob_v1': round(prob_v1 * 100, 1),
        'prob_e':  round(prob_e  * 100, 1),
        'prob_v2': round(prob_v2 * 100, 1),
        'placar_top': placar_top,
        'vencedor': vencedor,
    }
 

In [15]:
# =============================================================================
# 8. RODAR PREVISÕES PARA TODOS OS JOGOS
# =============================================================================
 
print("\n--- RODANDO PREVISÕES (Monte Carlo {:,} sim.) ---".format(N_SIMULACOES))
 
resultados = []
for _, row in df_jogos.iterrows():
    t1, t2 = row['Time1'], row['Time2']
    alt  = row['Altitude'] if pd.notna(row['Altitude']) else 5.0
    temp = row['Temp_Media'] if pd.notna(row['Temp_Media']) else 20.0
 
    pred = prever_jogo(t1, t2, alt, temp)
 
    top3 = pred['placar_top']
    placar1 = f"{top3[0][0][0]}-{top3[0][0][1]}" if top3 else "1-1"
    placar2 = f"{top3[1][0][0]}-{top3[1][0][1]}" if len(top3) > 1 else ""
    placar3 = f"{top3[2][0][0]}-{top3[2][0][1]}" if len(top3) > 2 else ""
 
    resultados.append({
        'Grupo':           row['Grupo'],
        'Data':            row['Data'],
        'Estádio':         row['Estadio'] or row['Cidade'],
        'Cidade':          row['Cidade'],
        'Altitude (m)':    round(alt, 0),
        'Temp. Média (°C)': round(temp, 1),
        'Time 1':          t1,
        'Time 2':          t2,
        'xG Esperado T1':  pred['lambda1'],
        'xG Esperado T2':  pred['lambda2'],
        '% Vitória T1':    pred['prob_v1'],
        '% Empate':        pred['prob_e'],
        '% Vitória T2':    pred['prob_v2'],
        'Placar Mais Provável': placar1,
        '2º Placar':       placar2,
        '3º Placar':       placar3,
        'Previsão Vencedor': pred['vencedor'],
    })
 
df_resultados = pd.DataFrame(resultados)
print(f"✓ {len(df_resultados)} jogos previstos")
print("\nAmostra (5 jogos):")
print(df_resultados[['Time 1','Time 2','% Vitória T1','% Empate','% Vitória T2','Previsão Vencedor']].head(5))
 


--- RODANDO PREVISÕES (Monte Carlo 50,000 sim.) ---
✓ 72 jogos previstos

Amostra (5 jogos):
             Time 1            Time 2  % Vitória T1  % Empate  % Vitória T2  \
0            México     África do Sul          27.8      42.5          29.7   
1     Coreia do Sul  República Tcheca          30.0      40.0          29.9   
2            México     Coreia do Sul          21.5      43.9          34.6   
3  República Tcheca     África do Sul          36.9      37.4          25.7   
4  República Tcheca            México          34.7      44.1          21.1   

  Previsão Vencedor  
0            Empate  
1            Empate  
2            Empate  
3            Empate  
4            Empate  


In [16]:
# =============================================================================
# 9. CLASSIFICAÇÃO SIMULADA DOS GRUPOS
# =============================================================================
# Para cada grupo, simular jogos e gerar tabela de pontos
 
def classificacao_grupo(grupo, df_res):
    jogos_g = df_res[df_res['Grupo'] == grupo]
    times = sorted(set(jogos_g['Time 1'].tolist() + jogos_g['Time 2'].tolist()))
    tabela = {t: {'Pts':0,'J':0,'V':0,'E':0,'D':0,'GP':0,'GC':0,'SG':0} for t in times}
 
    for _, r in jogos_g.iterrows():
        t1, t2 = r['Time 1'], r['Time 2']
        # Placar esperado (arredondado)
        g1 = round(r['xG Esperado T1'])
        g2 = round(r['xG Esperado T2'])
 
        tabela[t1]['J'] += 1; tabela[t2]['J'] += 1
        tabela[t1]['GP'] += g1; tabela[t1]['GC'] += g2
        tabela[t2]['GP'] += g2; tabela[t2]['GC'] += g1
 
        if g1 > g2:
            tabela[t1]['Pts'] += 3; tabela[t1]['V'] += 1; tabela[t2]['D'] += 1
        elif g1 == g2:
            tabela[t1]['Pts'] += 1; tabela[t1]['E'] += 1
            tabela[t2]['Pts'] += 1; tabela[t2]['E'] += 1
        else:
            tabela[t2]['Pts'] += 3; tabela[t2]['V'] += 1; tabela[t1]['D'] += 1
 
    for t in tabela:
        tabela[t]['SG'] = tabela[t]['GP'] - tabela[t]['GC']
 
    df_tab = pd.DataFrame(tabela).T.sort_values(['Pts','SG','GP'], ascending=False)
    df_tab.index.name = 'Seleção'
    df_tab['Grupo'] = grupo
    df_tab['Pos'] = range(1, len(df_tab)+1)
    return df_tab.reset_index()
 
dfs_tabelas = []
for g in sorted(df_resultados['Grupo'].unique()):
    dfs_tabelas.append(classificacao_grupo(g, df_resultados))
df_classificacao = pd.concat(dfs_tabelas, ignore_index=True)

In [17]:
# =============================================================================
# 10. RANKING GERAL DAS 48 SELEÇÕES
# =============================================================================
 
# Score: 60% xG, 40% defesa (inverso de xGC)
xg_max  = max(team_xg.values())
xgc_max = max(team_xgc.values())
 
ranking_rows = []
for t in df_teams.index:
    xg  = team_xg.get(t, 0)
    xgc = team_xgc.get(t, 1)
    score_atq = xg / xg_max
    score_def = 1 - (xgc / xgc_max)
    score_total = 0.6 * score_atq + 0.4 * score_def
 
    # Métricas reais
    row_data = df_teams.loc[t]
    ranking_rows.append({
        'Seleção':         t,
        'Continente':      continentes.get(t, '?'),
        'Score Total':     round(score_total * 100, 1),
        'Score Ataque':    round(score_atq  * 100, 1),
        'Score Defesa':    round(score_def  * 100, 1),
        'xG Médio':        round(xg, 3),
        'xGC Médio':       round(xgc, 3),
        'Gols Esperados (xG)':   round(float(row_data.get('Gols esperados (xG)', np.nan) or xg), 3),
        'xG Sofridos (xGC)':     round(float(row_data.get('xG sofridos (xGC)', np.nan) or xgc), 3),
        'Chances Perigosas':     row_data.get('Chances perigosas criadas', np.nan),
        'Posse de Bola (%)':     round(float(row_data.get('Posse de Bola', 0.5) or 0.5) * 100, 1),
        'Passes Completos':      row_data.get('Passes completos', np.nan),
        'Total Chutes':          row_data.get('Total de chutes', np.nan),
    })
 
df_ranking = pd.DataFrame(ranking_rows).sort_values('Score Total', ascending=False)
df_ranking['Rank'] = range(1, len(df_ranking)+1)
df_ranking = df_ranking[['Rank','Seleção','Continente','Score Total',
                          'Score Ataque','Score Defesa','xG Médio','xGC Médio',
                          'Chances Perigosas','Posse de Bola (%)','Total Chutes']]
 
print("\n--- TOP 10 RANKING GERAL ---")
print(df_ranking.head(10)[['Rank','Seleção','Score Total','xG Médio','xGC Médio']].to_string())


--- TOP 10 RANKING GERAL ---
    Rank               Seleção  Score Total  xG Médio  xGC Médio
21     1               Espanha         76.8     2.747      0.547
16     2               Croácia         70.0     2.740      0.791
8      3               Bélgica         69.3     2.777      0.843
38     4              Portugal         67.6     2.476      0.698
1      5              Alemanha         65.2     2.523      0.818
23     6                França         64.7     2.534      0.843
9      7  Bósnia e Herzegovina         63.7     3.186      1.331
27     8            Inglaterra         61.7     2.050      0.618
26     9               Holanda         57.2     2.109      0.826
34    10               Noruega         54.3     2.277      1.048


In [18]:
# =============================================================================
# 11. ANÁLISE DE CORRELAÇÃO
# =============================================================================
 
metricas_corr = ['Gols esperados (xG)', 'xG sofridos (xGC)',
                 'Chances perigosas criadas', 'Chutes no gol',
                 'Passes decisivos', 'Posse de Bola', 'Duelos ganhos',
                 'Assistência esperada (xA)']
df_corr_out = df_teams[metricas_corr].dropna().corr().round(3)

In [19]:
# =============================================================================
# 12. ESTATÍSTICA DESCRITIVA COMPLETA
# =============================================================================
 
df_desc_completa = df_teams.describe().T.reset_index()
df_desc_completa.columns = ['Métrica','N','Média','Std','Min','Q25','Mediana','Q75','Max']
df_desc_completa = df_desc_completa.round(3)

In [20]:
# =============================================================================
# 13. GERAR EXCEL COM OPENPYXL
# =============================================================================
 
print("\n--- GERANDO PLANILHA EXCEL ---")
 
wb = Workbook()
 
# --- Cores ---
COR_HEADER       = 'FF1B4F9A'  # azul escuro
COR_HEADER2      = 'FF2E75B6'  # azul médio
COR_VERDE        = 'FF70AD47'
COR_VERMELHO     = 'FFC00000'
COR_AMARELO      = 'FFFFE699'
COR_LARANJA      = 'FFF4B942'
COR_CINZA_CLARO  = 'FFF2F2F2'
COR_BRANCO       = 'FFFFFFFF'
COR_OURO         = 'FFFFD700'
 
def header_fill(cor):
    return PatternFill('solid', start_color=cor, end_color=cor)
 
def cell_fill(cor):
    return PatternFill('solid', start_color=cor, end_color=cor)
 
def thin_border():
    s = Side(style='thin', color='FFB8B8B8')
    return Border(left=s, right=s, top=s, bottom=s)
 
def write_df_to_sheet(ws, df, start_row=2, col_widths=None):
    """Escreve um DataFrame numa sheet com formatação profissional."""
    # Cabeçalhos
    for ci, col in enumerate(df.columns, 1):
        c = ws.cell(row=start_row, column=ci, value=col)
        c.font = Font(bold=True, color='FFFFFFFF', size=10, name='Arial')
        c.fill = header_fill(COR_HEADER)
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        c.border = thin_border()
 
    # Dados
    for ri, row_data in enumerate(df.itertuples(index=False), start_row+1):
        fill = cell_fill(COR_CINZA_CLARO) if ri % 2 == 0 else cell_fill(COR_BRANCO)
        for ci, val in enumerate(row_data, 1):
            c = ws.cell(row=ri, column=ci, value=val if not (isinstance(val, float) and np.isnan(val)) else '')
            c.fill = fill
            c.border = thin_border()
            c.alignment = Alignment(horizontal='center', vertical='center')
            c.font = Font(name='Arial', size=9)
 
    if col_widths:
        for ci, w in enumerate(col_widths, 1):
            ws.column_dimensions[get_column_letter(ci)].width = w
 
# ===== SHEET 1: CAPA / SUMÁRIO =====
ws_capa = wb.active
ws_capa.title = 'CAPA'
ws_capa.sheet_view.showGridLines = False
ws_capa.column_dimensions['A'].width = 5
ws_capa.column_dimensions['B'].width = 50
ws_capa.column_dimensions['C'].width = 30
 
# Título
ws_capa.row_dimensions[2].height = 50
c = ws_capa.cell(row=2, column=2, value='⚽  PREDIÇÕES COPA DO MUNDO 2026')
c.font = Font(bold=True, size=22, color='FF1B4F9A', name='Arial')
c.alignment = Alignment(horizontal='left', vertical='center')
 
c = ws_capa.cell(row=3, column=2, value='Modelo Estatístico | Regressão Linear + Distribuição de Poisson')
c.font = Font(size=12, color='FF666666', italic=True, name='Arial')
 
ws_capa.row_dimensions[5].height = 20
for r, txt in enumerate([
    ('METODOLOGIA', COR_HEADER, True),
    ('1.  Estatística Descritiva — médias, desvio padrão e distribuição de todas as 48 seleções', COR_BRANCO, False),
    ('2.  Regressão Linear Múltipla (Ridge) — xG e xGC previstos por métricas de desempenho', COR_CINZA_CLARO, False),
    ('3.  Ajuste Ambiental — penalização por altitude (>1500m) e temperatura extrema', COR_BRANCO, False),
    ('4.  Distribuição de Poisson — probabilidade de cada placar (0–6 gols)', COR_CINZA_CLARO, False),
    ('5.  Simulação Monte Carlo ({:,} iterações) — prob. de vitória, empate e derrota'.format(N_SIMULACOES), COR_BRANCO, False),
    ('', COR_BRANCO, False),
    ('ABAS DA PLANILHA', COR_HEADER, True),
    ('PREVISÕES JOGOS     — Todos os jogos da fase de grupos com probabilidades e placar previsto', COR_CINZA_CLARO, False),
    ('CLASSIFICAÇÃO       — Tabela de classificação simulada por grupo', COR_BRANCO, False),
    ('RANKING SELEÇÕES    — Ranking ofensivo/defensivo das 48 seleções', COR_CINZA_CLARO, False),
    ('STAT DESCRITIVA     — Estatística descritiva completa por métrica', COR_BRANCO, False),
    ('REGRESSÃO LINEAR    — Coeficientes e métricas do modelo de regressão', COR_CINZA_CLARO, False),
    ('CORRELAÇÕES         — Matriz de correlação entre métricas', COR_BRANCO, False),
    ('ESTÁDIOS            — Dados de altitude, temperatura e vento dos 16 estádios', COR_CINZA_CLARO, False),
], 7):
    ws_capa.row_dimensions[r].height = 18
    c = ws_capa.cell(row=r, column=2, value=txt[0])
    c.fill = cell_fill(txt[1])
    c.font = Font(bold=txt[2], size=10 if not txt[2] else 11, name='Arial',
                  color='FFFFFFFF' if txt[1] == COR_HEADER else 'FF333333')
    c.alignment = Alignment(horizontal='left', vertical='center', indent=1)
    c.border = thin_border()
 
# ===== SHEET 2: PREVISÕES DOS JOGOS =====
ws_pred = wb.create_sheet('PREVISÕES JOGOS')
ws_pred.sheet_view.showGridLines = False
ws_pred.freeze_panes = 'A3'
ws_pred.row_dimensions[1].height = 35
 
c = ws_pred.cell(row=1, column=1, value='⚽ PREVISÕES — FASE DE GRUPOS | COPA DO MUNDO 2026')
c.font = Font(bold=True, size=14, color='FFFFFFFF', name='Arial')
c.fill = header_fill(COR_HEADER)
c.alignment = Alignment(horizontal='center', vertical='center')
ws_pred.merge_cells('A1:R1')
 
write_df_to_sheet(ws_pred, df_resultados, start_row=2,
    col_widths=[5, 6, 22, 12, 8, 8, 18, 18, 8, 8, 8, 8, 8, 10, 9, 9, 9, 16])
 
# Colorir resultado: verde vitória T1, laranja empate, vermelho vitória T2
# Colunas: % Vitória T1 = col 11, % Empate = col 12, % Vitória T2 = col 13
# Vencedor = col 18
n_pred = len(df_resultados)
for ri in range(3, 3 + n_pred):
    row = df_resultados.iloc[ri - 3]
    venc = row['Previsão Vencedor']
    t1   = row['Time 1']
    t2   = row['Time 2']
    if venc == t1:
        cor = 'FFD5E8C4'  # verde claro
    elif venc == t2:
        cor = 'FFFFC7CE'  # vermelho claro
    else:
        cor = 'FFFFFFC7'  # amarelo claro
 
    for ci in range(7, 19):
        ws_pred.cell(row=ri, column=ci).fill = cell_fill(cor[2:])
 
# Ajustar: acima foi usado sem 'FF' prefixo — corrigindo
def color_rows(ws, n_rows, start_row=3, t1_col=7, t2_col=8, venc_col=18, df=None):
    for ri in range(start_row, start_row + n_rows):
        row_data = df.iloc[ri - start_row]
        venc = row_data['Previsão Vencedor']
        t1   = row_data['Time 1']
        t2   = row_data['Time 2']
        if venc == t1:
            cor = 'FFD5E8C4'
        elif venc == t2:
            cor = 'FFFFC7CE'
        else:
            cor = 'FFFFFF99'
        for ci in range(t1_col, venc_col + 1):
            ws.cell(row=ri, column=ci).fill = PatternFill('solid', start_color=cor, end_color=cor)
            ws.cell(row=ri, column=ci).border = thin_border()
 
color_rows(ws_pred, n_pred, df=df_resultados)
 
# ===== SHEET 3: CLASSIFICAÇÃO =====
ws_class = wb.create_sheet('CLASSIFICAÇÃO')
ws_class.sheet_view.showGridLines = False
 
c = ws_class.cell(row=1, column=1, value='📊 CLASSIFICAÇÃO SIMULADA — FASE DE GRUPOS')
c.font = Font(bold=True, size=14, color='FFFFFFFF', name='Arial')
c.fill = header_fill(COR_HEADER)
c.alignment = Alignment(horizontal='center', vertical='center')
ws_class.row_dimensions[1].height = 35
ws_class.merge_cells('A1:J1')
 
df_class_out = df_classificacao[['Grupo','Pos','Seleção','Pts','J','V','E','D','GP','GC','SG']]
write_df_to_sheet(ws_class, df_class_out, start_row=2,
    col_widths=[6,5,22,5,4,4,4,4,5,5,5])
 
# Colorir: 1º e 2º lugar verde (classificados), 3º amarelo (repescagem), 4º vermelho
cores_pos = {'1':'FF70AD47','2':'FF70AD47','3':'FFFFEB9C','4':'FFFFC7CE'}
for ri in range(3, 3 + len(df_class_out)):
    pos = str(df_class_out.iloc[ri-3]['Pos'])
    cor = cores_pos.get(pos, 'FFFFFFFF')
    for ci in range(1, 12):
        ws_class.cell(row=ri, column=ci).fill = PatternFill('solid', start_color=cor, end_color=cor)
        ws_class.cell(row=ri, column=ci).border = thin_border()
 
# ===== SHEET 4: RANKING =====
ws_rank = wb.create_sheet('RANKING SELEÇÕES')
ws_rank.sheet_view.showGridLines = False
c = ws_rank.cell(row=1, column=1, value='🏆 RANKING DAS 48 SELEÇÕES — SCORE COMPOSTO')
c.font = Font(bold=True, size=14, color='FFFFFFFF', name='Arial')
c.fill = header_fill(COR_HEADER)
c.alignment = Alignment(horizontal='center', vertical='center')
ws_rank.row_dimensions[1].height = 35
ws_rank.merge_cells('A1:K1')
 
write_df_to_sheet(ws_rank, df_ranking, start_row=2,
    col_widths=[5,22,12,10,10,10,8,8,10,10,10])
 
# Top 5 destacados em ouro
for ri in range(3, 8):
    for ci in range(1, 12):
        ws_rank.cell(row=ri, column=ci).fill = PatternFill('solid', start_color='FFFFF2CC', end_color='FFFFF2CC')
        ws_rank.cell(row=ri, column=ci).font  = Font(bold=True, name='Arial', size=9)
        ws_rank.cell(row=ri, column=ci).border = thin_border()
 
# ===== SHEET 5: ESTATÍSTICA DESCRITIVA =====
ws_desc = wb.create_sheet('STAT DESCRITIVA')
ws_desc.sheet_view.showGridLines = False
c = ws_desc.cell(row=1, column=1, value='📈 ESTATÍSTICA DESCRITIVA — 48 SELEÇÕES')
c.font = Font(bold=True, size=14, color='FFFFFFFF', name='Arial')
c.fill = header_fill(COR_HEADER)
c.alignment = Alignment(horizontal='center', vertical='center')
ws_desc.row_dimensions[1].height = 35
ws_desc.merge_cells('A1:I1')
 
write_df_to_sheet(ws_desc, df_desc_completa, start_row=2,
    col_widths=[30,5,8,8,8,8,8,8,8])
 
# ===== SHEET 6: REGRESSÃO LINEAR =====
ws_reg = wb.create_sheet('REGRESSÃO LINEAR')
ws_reg.sheet_view.showGridLines = False
ws_reg.column_dimensions['A'].width = 32
ws_reg.column_dimensions['B'].width = 14
ws_reg.column_dimensions['C'].width = 14
ws_reg.column_dimensions['D'].width = 30
 
c = ws_reg.cell(row=1, column=1, value='📐 REGRESSÃO LINEAR MÚLTIPLA (Ridge Regression)')
c.font = Font(bold=True, size=14, color='FFFFFFFF', name='Arial')
c.fill = header_fill(COR_HEADER)
c.alignment = Alignment(horizontal='center', vertical='center')
ws_reg.row_dimensions[1].height = 35
ws_reg.merge_cells('A1:D1')
 
# Sumário do modelo
info_modelo = [
    ['','','',''],
    ['SUMÁRIO DO MODELO','','',''],
    ['Variável Resposta (Ataque)','xG esperado por jogo','',''],
    ['Variável Resposta (Defesa)','xGC sofrido por jogo','',''],
    ['Tipo de Modelo','Ridge Regression (α=1.0)','',''],
    ['Normalização','StandardScaler (z-score)','',''],
    [f'R² — Modelo Ataque (xG)',  f'{r2_atq:.4f}', '', ''],
    [f'R² — Modelo Defesa (xGC)', f'{r2_def:.4f}', '', ''],
    ['Nº de observações', str(len(df_model)), '', ''],
    ['','','',''],
    ['COEFICIENTES DO MODELO','','',''],
    ['Feature','Coef. xG (Ataque)','Coef. xGC (Defesa)','Interpretação'],
]
for r, vals in enumerate(info_modelo, 2):
    ws_reg.row_dimensions[r].height = 18
    for ci, v in enumerate(vals, 1):
        c = ws_reg.cell(row=r, column=ci, value=v)
        if vals[0] in ('SUMÁRIO DO MODELO','COEFICIENTES DO MODELO',''):
            c.font = Font(bold=True, size=10, name='Arial',
                          color='FFFFFFFF' if vals[0] in ('SUMÁRIO DO MODELO','COEFICIENTES DO MODELO') else 'FF333333')
            if vals[0] in ('SUMÁRIO DO MODELO','COEFICIENTES DO MODELO'):
                c.fill = header_fill(COR_HEADER2)
        else:
            c.font = Font(size=10, name='Arial')
            c.fill = cell_fill(COR_CINZA_CLARO if r % 2 == 0 else COR_BRANCO)
        c.border = thin_border()
        c.alignment = Alignment(horizontal='left', vertical='center', indent=1)
 
interpretacoes = {
    'Chances perigosas criadas': 'Finalizações de alta qualidade → ↑ xG',
    'Chutes no gol':             'Chutes no alvo → ↑ xG diretamente',
    'Passes decisivos':          'Passes que criam chance → ↑ xG',
    'Assistência esperada (xA)': 'Qualidade das assistências → ↑ xG',
    'Passes no último terço':    'Chegada na área → ↑ pressão ofensiva',
    'Posse de Bola':             'Controle do jogo → ↑ xG moderado',
    'Total de chutes':           'Volume de finalizações → ↑ xG',
    'Chutes dentro da área':     'Finalizações próximas → alto impacto',
}
 
for ri, row_c in df_coef.iterrows():
    r = 14 + ri
    ws_reg.row_dimensions[r].height = 18
    vals = [row_c['Feature'], row_c['Coef_xG'], row_c['Coef_xGC'],
            interpretacoes.get(row_c['Feature'], '')]
    for ci, v in enumerate(vals, 1):
        c = ws_reg.cell(row=r, column=ci, value=v)
        c.font = Font(size=10, name='Arial')
        c.fill = cell_fill(COR_CINZA_CLARO if r % 2 == 0 else COR_BRANCO)
        c.border = thin_border()
        c.alignment = Alignment(horizontal='left' if ci in (1,4) else 'center',
                                 vertical='center', indent=1)
        if ci in (2, 3) and isinstance(v, float):
            c.number_format = '0.0000'
            if v > 0:
                c.font = Font(size=10, name='Arial', color='FF375623', bold=True)
            elif v < 0:
                c.font = Font(size=10, name='Arial', color='FF8B0000')
 
# ===== SHEET 7: CORRELAÇÕES =====
ws_corr = wb.create_sheet('CORRELAÇÕES')
ws_corr.sheet_view.showGridLines = False
c = ws_corr.cell(row=1, column=1, value='🔗 MATRIZ DE CORRELAÇÃO — MÉTRICAS PRINCIPAIS')
c.font = Font(bold=True, size=14, color='FFFFFFFF', name='Arial')
c.fill = header_fill(COR_HEADER)
c.alignment = Alignment(horizontal='center', vertical='center')
ws_corr.row_dimensions[1].height = 35
n_m = len(df_corr_out)
ws_corr.merge_cells(f'A1:{get_column_letter(n_m + 1)}1')
 
# Cabeçalho de colunas
labels = list(df_corr_out.columns)
for ci, lbl in enumerate(labels, 2):
    c = ws_corr.cell(row=2, column=ci, value=lbl)
    c.font = Font(bold=True, size=8, color='FFFFFFFF', name='Arial')
    c.fill = header_fill(COR_HEADER2)
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    c.border = thin_border()
    ws_corr.column_dimensions[get_column_letter(ci)].width = 14
ws_corr.column_dimensions['A'].width = 28
ws_corr.row_dimensions[2].height = 40
 
for ri, (idx, row_c) in enumerate(df_corr_out.iterrows(), 3):
    ws_corr.row_dimensions[ri].height = 18
    c = ws_corr.cell(row=ri, column=1, value=idx)
    c.font = Font(bold=True, size=9, name='Arial')
    c.fill = header_fill(COR_HEADER2)
    c.font = Font(bold=True, size=9, color='FFFFFFFF', name='Arial')
    c.border = thin_border()
    c.alignment = Alignment(horizontal='left', vertical='center', indent=1)
 
    for ci, v in enumerate(row_c, 2):
        c = ws_corr.cell(row=ri, column=ci, value=round(v, 3))
        c.number_format = '0.000'
        c.border = thin_border()
        c.alignment = Alignment(horizontal='center', vertical='center')
        # Heatmap de cores
        if abs(v) >= 0.9:
            cor = 'FF1F5C2E' if v > 0 else 'FF8B0000'
            c.font = Font(bold=True, size=9, color='FFFFFFFF', name='Arial')
            c.fill = PatternFill('solid', start_color=cor, end_color=cor)
        elif abs(v) >= 0.7:
            cor = 'FF70AD47' if v > 0 else 'FFFF6B6B'
            c.font = Font(bold=True, size=9, name='Arial')
            c.fill = PatternFill('solid', start_color=cor, end_color=cor)
        elif abs(v) >= 0.5:
            cor = 'FFCCE5CC' if v > 0 else 'FFFFD5D5'
            c.font = Font(size=9, name='Arial')
            c.fill = PatternFill('solid', start_color=cor, end_color=cor)
        else:
            c.font = Font(size=9, name='Arial')
            c.fill = cell_fill(COR_CINZA_CLARO)
 
# ===== SHEET 8: ESTÁDIOS =====
ws_est = wb.create_sheet('ESTÁDIOS')
ws_est.sheet_view.showGridLines = False
c = ws_est.cell(row=1, column=1, value='🏟️ ESTÁDIOS — DADOS AMBIENTAIS COPA 2026')
c.font = Font(bold=True, size=14, color='FFFFFFFF', name='Arial')
c.fill = header_fill(COR_HEADER)
c.alignment = Alignment(horizontal='center', vertical='center')
ws_est.row_dimensions[1].height = 35
ws_est.merge_cells('A1:H1')
 
df_est_out = df_est[['ESTADIO','CIDADE','PAÍS','ALTITUDE (m)','temp_media_calc',
                      'Velocidade do vento a 10 metros','LATITUDE','LONGITUDE']].copy()
df_est_out.columns = ['Estádio','Cidade','País','Altitude (m)','Temp. Média (°C)',
                       'Vento Médio (km/h)','Latitude','Longitude']
df_est_out = df_est_out.sort_values('Altitude (m)', ascending=False).reset_index(drop=True)
df_est_out['Temp. Média (°C)'] = df_est_out['Temp. Média (°C)'].round(1)
 
write_df_to_sheet(ws_est, df_est_out, start_row=2,
    col_widths=[26,18,10,12,14,16,12,12])
 
# Estádio Azteca destacado (maior altitude)
for ci in range(1, 9):
    ws_est.cell(row=3, column=ci).fill = PatternFill('solid', start_color='FFFFF2CC', end_color='FFFFF2CC')
    ws_est.cell(row=3, column=ci).font = Font(bold=True, size=9, name='Arial')
    ws_est.cell(row=3, column=ci).border = thin_border()
 
# ===== LEGENDA / NOTA RODAPÉ em cada sheet =====
def add_footer(ws, row, text, col=1):
    c = ws.cell(row=row+2, column=col, value=text)
    c.font = Font(italic=True, size=8, color='FF888888', name='Arial')
    c.alignment = Alignment(horizontal='left', vertical='center')
 
add_footer(ws_pred, 3+n_pred,
    '💡 Verde = Favoritismo Time 1 | Vermelho = Favoritismo Time 2 | Amarelo = Empate provável | '
    'Previsões baseadas em Poisson + Monte Carlo ({:,} simulações)'.format(N_SIMULACOES))
add_footer(ws_class, 3+len(df_class_out),
    '💡 Verde = Classificados (1º e 2º) | Amarelo = Possível repescagem (3º) | Vermelho = Eliminados (4º)')
add_footer(ws_corr, 3+len(df_corr_out),
    '💡 Verde escuro = correlação muito forte (≥0.9) | Verde = forte (≥0.7) | Verde claro = moderada (≥0.5) | Vermelho = correlação negativa')


--- GERANDO PLANILHA EXCEL ---


In [21]:
# =============================================================================
# 14. SALVAR
# =============================================================================
wb.save(ARQUIVO_SAIDA)
print(f"\n✅ PLANILHA SALVA: {ARQUIVO_SAIDA}")
print("=" * 70)
print("RESUMO DAS PREVISÕES:")
print(f"  Jogos previstos:     {len(df_resultados)}")
print(f"  Seleções analisadas: {len(df_teams)}")
print(f"  Simulações MC:       {N_SIMULACOES:,}")
print(f"  R² modelo xG:        {r2_atq:.4f}")
print(f"  R² modelo xGC:       {r2_def:.4f}")
print("=" * 70)
print("\nTOP 5 DO RANKING:")
for _, r in df_ranking.head(5).iterrows():
    print(f"  {int(r['Rank'])}. {r['Seleção']:<22} Score: {r['Score Total']:.1f}  xG: {r['xG Médio']:.3f}  xGC: {r['xGC Médio']:.3f}")
print("\nBRASIL nos jogos:")
for _, r in df_resultados[df_resultados['Time 1'].eq('Brasil') | df_resultados['Time 2'].eq('Brasil')].iterrows():
    print(f"  {r['Time 1']:18} vs {r['Time 2']:18}  → {r['Previsão Vencedor']:20}  "
          f"({r['% Vitória T1']}% / {r['% Empate']}% / {r['% Vitória T2']}%)  Placar: {r['Placar Mais Provável']}")
print("=" * 70)


✅ PLANILHA SALVA: C:/Users/raphael.eugenio/Desktop/Raphael/WC26/Scripts/predicoes_copa2026_resultados.xlsx
RESUMO DAS PREVISÕES:
  Jogos previstos:     72
  Seleções analisadas: 48
  Simulações MC:       50,000
  R² modelo xG:        0.9486
  R² modelo xGC:       0.3728

TOP 5 DO RANKING:
  1. Espanha                Score: 76.8  xG: 2.747  xGC: 0.547
  2. Croácia                Score: 70.0  xG: 2.740  xGC: 0.791
  3. Bélgica                Score: 69.3  xG: 2.777  xGC: 0.843
  4. Portugal               Score: 67.6  xG: 2.476  xGC: 0.698
  5. Alemanha               Score: 65.2  xG: 2.523  xGC: 0.818

BRASIL nos jogos:
  Brasil             vs Marrocos            → Brasil                (36.8% / 35.9% / 27.3%)  Placar: 0-0
  Brasil             vs Haiti               → Brasil                (55.2% / 39.1% / 5.7%)  Placar: 0-0
  Escócia            vs Brasil              → Empate                (32.8% / 39.6% / 27.7%)  Placar: 0-0
